# DanmakuRender：按运行编号补渲染、归档与投稿

运行后先挂载 Drive，再填写历史 `run_id`（留空则自动选择 Drive 中最近一次失败/待恢复的任务）并从上到下运行，或直接点击「运行时 → 全部运行」。先校验 Drive 清单与文件，复用已完成成品及合并，缺失成品从原片和 ASS 补渲染，再补做合并归档和多 P 投稿。原录制错误保留，无法补回未录到的内容。

开始前结束该编号的录制、补传及其他恢复实例。建议使用独立 GPU 运行时，不与正在录制的 Notebook 共用。投稿使用私有 Drive 的 `DMRColab/config/upload.yml` 和 `credentials/bilibili_upload.json`，不要在本 Notebook 粘贴 Cookie。`unknown` 只核对远端，无法确认时停止。

源码需包含 `colab_resume.py`；私有副本不会自动随仓库更新。单任务结果保存到 `resume_manifest.json`；跨任务追加结果保存到主任务的 `append_resume.json`。默认开启 `AUTO_RELEASE_RUNTIME=True`，在执行完毕且结果保存到 Drive 后自动调用 `runtime.unassign()` 断开并删除运行时。

跨任务追加：`RUN_ID` 填已创建稿件的主任务，`APPEND_RUN_IDS` 填同场后续任务（多个用英文逗号分隔），`TARGET_BVID` 填主任务原稿件 BV 号，保持 `UPLOAD=True`、`RECONCILE_BVID` 留空。先检查第④步的顺序、重复及时间间隙，再执行第⑤步。跨任务模式只恢复分段并追加分 P，不生成跨任务合并归档。


In [ ]:
#@title ① 挂载私有 Drive（第一步）
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
RESUME_PHASE = 'drive'
RESUME_PARAMETERS = None
RESUME_COMMAND = None
print('Drive 已挂载；现在填写要恢复的 run_id（留空将自动匹配最近一次失败任务）。')


In [ ]:
#@title ② 选择历史任务（RUN_ID 留空将自动选择最近一次失败任务）
previous_phase = globals().get('RESUME_PHASE')
RESUME_PHASE = 'invalid'
RESUME_PARAMETERS = None
RESUME_COMMAND = None
RUN_ID = '' #@param {type:"string"}
APPEND_RUN_IDS = '' #@param {type:"string"}
TARGET_BVID = '' #@param {type:"string"}
UPLOAD = True #@param {type:"boolean"}
RECONCILE_BVID = '' #@param {type:"string"}
ENCODER = 'h264_nvenc' #@param ["h264_nvenc", "libx264"]
MEDIA_TIMEOUT_HOURS = 4 #@param {type:"number"}
MIN_FREE_GIB = 10 #@param {type:"number"}
AUTO_RELEASE_RUNTIME = True #@param {type:"boolean"}
import json, math, re
if previous_phase != 'drive':
    raise RuntimeError('请先运行第一步并成功挂载 Drive')
selected_run_id = RUN_ID.strip()
if not selected_run_id:
    runs_dir = DRIVE_ROOT / 'DMRColab/runs'
    if runs_dir.is_dir():
        for candidate in sorted((p.name for p in runs_dir.iterdir() if p.is_dir() and re.fullmatch(r'[0-9]{8}-[0-9]{6}-[0-9a-f]{8}', p.name)), reverse=True):
            m_path = runs_dir / candidate / 'manifest.json'
            r_path = runs_dir / candidate / 'resume_manifest.json'
            if not m_path.is_file():
                continue
            try:
                m_data = json.loads(m_path.read_text(encoding='utf-8'))
                r_data = json.loads(r_path.read_text(encoding='utf-8')) if r_path.is_file() else {}
            except Exception:
                continue
            if m_data.get('status') == 'failed' and r_data.get('status') != 'success':
                selected_run_id = candidate
                print('未填写 RUN_ID，已自动选择最近待恢复任务:', selected_run_id)
                break
if not re.fullmatch(r'[0-9]{8}-[0-9]{6}-[0-9a-f]{8}', selected_run_id):
    raise ValueError('请填写完整运行编号，例如 20261001-220500-bbbbbbbb（虚构示例）')
if RECONCILE_BVID.strip() and not re.fullmatch(r'BV[0-9A-Za-z]{10}', RECONCILE_BVID.strip()):
    raise ValueError('候选 BVID 格式无效；通常留空')
append_ids = [value.strip() for value in APPEND_RUN_IDS.split(',') if value.strip()]
if any(not re.fullmatch(r'[0-9]{8}-[0-9]{6}-[0-9a-f]{8}', value) for value in append_ids):
    raise ValueError('追加任务编号格式无效，请用逗号分隔完整 run_id')
if TARGET_BVID.strip() and not re.fullmatch(r'BV[0-9A-Za-z]{10}', TARGET_BVID.strip()):
    raise ValueError('目标 BV 号格式无效')
if bool(append_ids) != bool(TARGET_BVID.strip()):
    raise ValueError('跨任务追加必须同时填写 APPEND_RUN_IDS 和 TARGET_BVID')
if len(set([selected_run_id, *append_ids])) != len(append_ids) + 1:
    raise ValueError('主任务与追加任务编号不能重复')
if append_ids and (UPLOAD is not True or RECONCILE_BVID.strip()):
    raise ValueError('跨任务追加需要 UPLOAD=True，RECONCILE_BVID 留空')
if ENCODER not in ('h264_nvenc', 'libx264') or type(UPLOAD) is not bool or type(AUTO_RELEASE_RUNTIME) is not bool:
    raise ValueError('参数无效')
if any(type(v) not in (int, float) or not math.isfinite(v) or v <= 0 for v in (MEDIA_TIMEOUT_HOURS, MIN_FREE_GIB)):
    raise ValueError('时限与磁盘阈值必须为有限正数')
RESUME_PARAMETERS = dict(run_id=selected_run_id, append_ids=append_ids, target_bvid=TARGET_BVID.strip(), upload=UPLOAD, bvid=RECONCILE_BVID.strip(), encoder=ENCODER, timeout=MEDIA_TIMEOUT_HOURS * 3600, free=MIN_FREE_GIB, auto_release=AUTO_RELEASE_RUNTIME)
BACKUP = DRIVE_ROOT / 'DMRColab/runs' / RESUME_PARAMETERS['run_id']
if any(not (DRIVE_ROOT / 'DMRColab/runs' / value / 'manifest.json').is_file() for value in [RESUME_PARAMETERS['run_id'], *append_ids]):
    raise RuntimeError('Drive 中未找到某个运行编号的 manifest.json')
RESUME_PHASE = 'form'


In [ ]:
#@title ③ 下载本次恢复源码并安装依赖
previous_phase = globals().get('RESUME_PHASE')
RESUME_PHASE = 'invalid'
RESUME_COMMAND = None
if previous_phase != 'form':
    raise RuntimeError('请先填写并成功运行历史任务表单')
import subprocess, sys, tempfile
PROJECT = Path(tempfile.mkdtemp(prefix='dmr-recovery-', dir='/content')) / 'project'
def checked(command, **kwargs):
    return subprocess.run(command, check=True, **kwargs)
checked(['git', 'clone', '--depth', '1', '--branch', 'codex/colab-poc', 'https://github.com/zaixiakongyiji/DanmakuRenderForColab.git', str(PROJECT)])
if not (PROJECT / 'colab_resume.py').is_file():
    raise RuntimeError('仓库分支尚未发布恢复入口，请先提交并推送恢复代码')
merge_file = PROJECT / 'colab_support/merge.py'
merge_text = merge_file.read_text(encoding='utf-8')
if '_effective_video_frame_rate' not in merge_text:
    old_block = (
        "        signature = [{key: s.get(key) for key in fields} for s in streams]\n"
        "        if any(not s.get('codec_name') or not s.get('time_base') for s in streams):\n"
        "            raise ValueError()\n"
        "    except (ValueError, KeyError, TypeError):\n"
        "        raise MergeError('merge_invalid_media') from None\n"
        "    return duration, signature\n\n\n"
        "def video_frame_rate(signature):\n"
        "    try:\n"
        "        video = next(s for s in signature if s.get('codec_type') == 'video')\n"
        "        rate = Fraction(video['r_frame_rate'])\n"
        "        if not 1 <= rate <= 120:\n"
        "            raise ValueError()\n"
        "        return str(rate)\n"
        "    except (StopIteration, KeyError, TypeError, ValueError, ZeroDivisionError):\n"
        "        raise MergeError('merge_invalid_frame_rate') from None"
    )
    new_block = (
        "        signature = []\n"
        "        for s in streams:\n"
        "            item = {key: s.get(key) for key in fields}\n"
        "            if s.get('codec_type') == 'video':\n"
        "                try:\n"
        "                    effective = _effective_video_frame_rate(s)\n"
        "                    if _parse_frame_rate(s.get('r_frame_rate')) != effective:\n"
        "                        item['r_frame_rate'] = f'{effective.numerator}/{effective.denominator}'\n"
        "                except ValueError:\n"
        "                    pass\n"
        "            signature.append(item)\n"
        "        if any(not s.get('codec_name') or not s.get('time_base') for s in streams):\n"
        "            raise ValueError()\n"
        "    except (ValueError, KeyError, TypeError):\n"
        "        raise MergeError('merge_invalid_media') from None\n"
        "    return duration, signature\n\n\n"
        "def _parse_frame_rate(value):\n"
        "    if not value or not isinstance(value, str):\n"
        "        return None\n"
        "    try:\n"
        "        rate = Fraction(value)\n"
        "    except (ValueError, ZeroDivisionError):\n"
        "        return None\n"
        "    return rate if rate > 0 else None\n\n\n"
        "def _effective_video_frame_rate(video):\n"
        "    r_rate = _parse_frame_rate(video.get('r_frame_rate'))\n"
        "    avg_rate = _parse_frame_rate(video.get('avg_frame_rate'))\n"
        "    if r_rate is not None and 1 <= r_rate <= 120:\n"
        "        if avg_rate is None or not (1 <= avg_rate <= 120) or abs(r_rate - avg_rate) / avg_rate <= Fraction(1, 5):\n"
        "            return r_rate\n"
        "    if avg_rate is not None and 1 <= avg_rate <= 120:\n"
        "        common = (15, 24, 25, 30, 48, 50, 60, 90, 120)\n"
        "        nearest = min(common, key=lambda target: abs(avg_rate - target))\n"
        "        snapped = Fraction(nearest, 1) if abs(avg_rate - nearest) <= Fraction(3, 10) else avg_rate.limit_denominator(1001)\n"
        "        if 1 <= snapped <= 120:\n"
        "            return snapped\n"
        "    raise ValueError('invalid_frame_rate')\n\n\n"
        "def video_frame_rate(signature):\n"
        "    try:\n"
        "        video = next(s for s in signature if s.get('codec_type') == 'video')\n"
        "        rate = _effective_video_frame_rate(video)\n"
        "        return str(rate)\n"
        "    except (StopIteration, KeyError, TypeError, ValueError, ZeroDivisionError):\n"
        "        raise MergeError('merge_invalid_frame_rate') from None"
    )
    if old_block in merge_text:
        merge_file.write_text(merge_text.replace(old_block, new_block, 1), encoding='utf-8')
        print('已自动为云端检出注入高帧率抖动兼容补丁。')
print('源码 commit:', subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip())
if RESUME_PARAMETERS['append_ids']:
    import ast
    tree = ast.parse((PROJECT / 'colab_resume.py').read_text(encoding='utf-8'))
    options = {arg.value for node in ast.walk(tree) if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr == 'add_argument' for arg in node.args if isinstance(arg, ast.Constant) and isinstance(arg.value, str)}
    if not {'--append-run-id', '--target-bvid'} <= options or not (PROJECT / 'colab_support/session_append.py').is_file():
        raise RuntimeError('下载的源码尚不支持跨任务追加。请将本地新增代码提交并推送到 colab 远程的 codex/colab-poc 分支，再重新运行第①②③步下载源码。')
checked(['apt-get', 'update', '-qq'])
checked(['apt-get', 'install', '-y', 'ffmpeg', 'fontconfig', 'fonts-noto-cjk'])
checked([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT / 'colab_support/requirements.txt')])
checked(['fc-cache', '-f'])
p = RESUME_PARAMETERS
RESUME_COMMAND = (sys.executable, str(PROJECT / 'colab_resume.py'), '--run-id', p['run_id'], '--drive-root', str(DRIVE_ROOT), '--encoder', p['encoder'], '--media-timeout', str(p['timeout']), '--min-free-gib', str(p['free']))
if not p['upload']:
    RESUME_COMMAND += ('--no-upload',)
if p['append_ids']:
    RESUME_COMMAND += tuple(sum((['--append-run-id', value] for value in p['append_ids']), [])) + ('--target-bvid', p['target_bvid'])
if p['bvid']:
    RESUME_COMMAND += ('--reconcile-bvid', p['bvid'])
RESUME_PHASE = 'dependencies'


In [ ]:
#@title ④ 校验备份并显示恢复步骤
previous_phase = globals().get('RESUME_PHASE')
RESUME_PHASE = 'invalid'
if previous_phase != 'dependencies' or globals().get('RESUME_COMMAND') is None:
    raise RuntimeError('请先成功完成源码和依赖准备')
inspection = subprocess.run([*RESUME_COMMAND, '--inspect-only'], cwd=PROJECT, capture_output=True, text=True, check=False)
if inspection.stdout:
    print(inspection.stdout, end='')
if inspection.stderr:
    print(inspection.stderr, end='')
if inspection.returncode:
    message = '源码参数不兼容，请确认新增代码已推送并重新下载源码' if inspection.returncode == 2 else '备份校验未通过，请查看上方恢复错误代码'
    raise RuntimeError(f'{message}（退出码 {inspection.returncode}）；尚未开始恢复与投稿。')
RESUME_PHASE = 'inspected'


In [ ]:
#@title ⑤ 继续处理与投稿（完成后默认自动释放运行时）
previous_phase = globals().get('RESUME_PHASE')
RESUME_PHASE = 'invalid'
if previous_phase != 'inspected' or globals().get('RESUME_COMMAND') is None:
    raise RuntimeError('请先成功完成备份校验')
RESUME_PHASE = 'running'
try:
    result = subprocess.run(list(RESUME_COMMAND), cwd=PROJECT, check=False)
    print('恢复退出码:', result.returncode)
finally:
    RESUME_PHASE = 'finished'
    import json
    summary = BACKUP / ('append_resume.json' if RESUME_PARAMETERS['append_ids'] else 'resume_manifest.json')
    if summary.is_file():
        state = json.loads(summary.read_text(encoding='utf-8'))
        print(json.dumps({k: state.get(k) for k in ('run_id', 'bvid', 'status', 'phase', 'error', 'original_status', 'original_errors', 'errors', 'merge', 'upload')}, ensure_ascii=False, indent=2))
    print('结果目录:', BACKUP, flush=True)
    if RESUME_PARAMETERS.get('auto_release', True) and summary.is_file():
        print('恢复结果已保存到 Drive；正在请求释放并删除 Colab 运行时...', flush=True)
        from google.colab import runtime
        runtime.unassign()
    else:
        print('运行时保留。检查结果后请自行断开并删除运行时；重试请重新运行表单及后续步骤。', flush=True)
